In [1]:
import datetime
import logging  # 追加：ログ制御用
import warnings  # 追加：警告非表示用
import pandas as pd
import yfinance as yf
from IPython.display import display, HTML

# 警告メッセージ（FutureWarningなど）を非表示にする
warnings.filterwarnings("ignore")

# yfinanceの内部エラー出力を非表示にする
yf_logger = logging.getLogger('yfinance')
yf_logger.setLevel(logging.CRITICAL)

# Define the lookback periods (in business days) for performance calculation.
LOOKBACK_DAYS = {
    "1 Day (%)": -2,
    "3 Days (%)": -4,
    "1 Week (%)": -6,
    "1 Month (%)": -21,
    "6 Months (%)": -126,
    "1 Year (%)": -252,
}

# =====================================================================
# DATA PROCESSING FUNCTION (Supports Market Segmentation)
# =====================================================================
def show_market_performances(market_dict):
    today = datetime.date.today()
    start_date = today - datetime.timedelta(days=60)
    
    # Format numerical values to 2 decimal places
    format_config = {"Latest Price": "{:,.2f}"}
    for column_name in LOOKBACK_DAYS.keys():
        format_config[column_name] = "{:+.2f}%"
        
    # ハイライト用の条件関数
    def highlight_extreme_values(val):
        if pd.isna(val):
            return ''
        if val >= 10:
            return 'background-color: #ffcccc; color: #cc0000; font-weight: bold;'
        elif val <= -10:
            return 'background-color: #cce5ff; color: #004085; font-weight: bold;'
        return ''
        
    # Process each market section
    for market_name, ticker_dict in market_dict.items():
        records = []
        
        # Display market header
        display(HTML(f"<h3 style='margin-top: 20px; color: #333;'>■ {market_name} Market</h3>"))
        
        for name, ticker in ticker_dict.items():
            try:
                # 【修正】 ignore_tz=True を追加してタイムゾーン警告を抑制
                df = yf.download(ticker, start=start_date, end=today, progress=False, ignore_tz=True)
                
                if df.empty:
                    # 元のコードにあったWarning表示も邪魔ならコメントアウトしてください
                    # print(f"[Warning] No data found for: {name} ({ticker})")
                    continue
                    
                prices = df["Close"].squeeze()
                latest_price = prices.iloc[-1]
                
                row = {
                    "Name": name,
                    "Ticker": ticker,
                    "Latest Price": latest_price
                }
                
                for label, index in LOOKBACK_DAYS.items():
                    if len(prices) >= abs(index):
                        past_price = prices.iloc[index]
                    else:
                        past_price = prices.iloc[0]
                    
                    row[label] = ((latest_price - past_price) / past_price) * 100
                    
                records.append(row)
                
            except Exception as e:
                # エラー時のprintも非表示にしたい場合はコメントアウトしてください
                pass
        
        if records:
            df_performance = pd.DataFrame(records)
            
            # .map を使用
            styled_df = (df_performance.style
                         .format(format_config)
                         .map(highlight_extreme_values, subset=list(LOOKBACK_DAYS.keys()))
                         .hide(axis="index"))
            
            display(styled_df)
        else:
            print("No data available for this market.")

In [2]:
# =====================================================================
# CONFIGURATION & EXECUTION (Segmented by Market)
# =====================================================================
MARKET_TICKERS = {
    "United States": {
        "S&P 500": "^GSPC",
        "NASDAQ Composite": "^IXIC",
        "Apple": "AAPL",
        "NVIDIA": "NVDA",
        "Microsoft": "MSFT",
    },
    "Japan": {
        "Nikkei 225": "^N225",
        "Toyota Motor": "7203.T",
        "Sony Group": "6758.T",
        "Mitsubishi UFJ Financial": "8306.T",
    },
    "Singapore": {
        "STI Index": "^STI",
        "DBS Group": "D05.SI",
        "UOB": "U11.SI",
        "Singtel": "Z74.SI",
        "CapitaLand Ascendas REIT": "A17U.SI",
    },
    "Japan Stock": {
        "小松製作所 (6301)": "6301.T",
        "ニデック (6594)": "6594.T",
        "未来工業 (7931)": "7931.T",
        "東部ネットワーク (9036)": "9036.T",
        "ニトリHD (9843)": "9843.T",
        "MRK HD (9980)": "9980.T",
    },
    "US Stock (ETF)": {
        "iシェアーズ コア米国総合債券ETF (AGG)": "AGG"
    }
}

# Run and display the segmented tables
show_market_performances(MARKET_TICKERS)

Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
S&P 500,^GSPC,"7,575.39",+0.42%,+0.95%,+1.23%,+4.24%,+0.99%,+0.99%
NASDAQ Composite,^IXIC,"26,281.61",+0.29%,+1.79%,+1.74%,+4.42%,-1.33%,-1.33%
Apple,AAPL,315.32,-0.28%,+1.50%,+2.17%,+8.14%,+5.74%,+5.74%
NVIDIA,NVDA,210.96,+4.03%,+7.12%,+8.28%,+5.26%,-10.41%,-10.41%
Microsoft,MSFT,385.10,+0.19%,-0.96%,-1.38%,-3.09%,-5.74%,-5.74%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Nikkei 225,^N225,"68,557.73",+1.20%,+0.44%,-1.70%,+3.84%,+9.42%,+9.42%
Toyota Motor,7203.T,"2,823.00",-0.04%,-4.18%,-0.18%,+1.71%,-6.15%,-6.15%
Sony Group,6758.T,"3,359.00",-1.47%,-4.08%,-0.62%,+2.04%,-2.47%,-2.47%
Mitsubishi UFJ Financial,8306.T,"3,461.00",+1.38%,+0.44%,+4.06%,+9.46%,+20.30%,+20.30%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
STI Index,^STI,"5,469.29",+0.65%,+2.38%,+4.29%,+8.82%,+9.47%,+9.47%
DBS Group,D05.SI,70.45,+0.31%,+2.64%,+5.53%,+11.40%,+17.16%,+17.16%
UOB,U11.SI,44.38,+0.09%,+6.45%,+10.29%,+16.30%,+18.76%,+18.76%
Singtel,Z74.SI,4.40,+0.23%,+0.46%,-1.57%,+3.04%,-9.65%,-9.65%
CapitaLand Ascendas REIT,A17U.SI,2.52,+1.20%,+0.80%,+1.20%,-0.40%,+2.02%,+2.02%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
小松製作所 (6301),6301.T,"6,395.00",+0.00%,-4.69%,-3.50%,-1.99%,-2.52%,-2.52%
ニデック (6594),6594.T,"2,660.00",+1.80%,+0.15%,-4.56%,+1.26%,-1.00%,-1.00%
未来工業 (7931),7931.T,"3,215.00",+0.16%,+0.00%,+2.23%,+5.41%,+6.63%,+6.63%
東部ネットワーク (9036),9036.T,"1,265.00",-1.79%,-1.02%,-1.02%,+7.20%,+2.02%,+2.02%
ニトリHD (9843),9843.T,"2,380.00",-1.90%,-3.43%,+1.73%,-11.54%,+5.08%,+5.08%
MRK HD (9980),9980.T,96.00,+0.00%,+0.00%,+2.13%,-1.03%,-1.03%,-1.03%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
iシェアーズ コア米国総合債券ETF (AGG),AGG,98.08,-0.10%,-0.13%,-0.54%,+0.10%,+0.10%,+0.10%


In [3]:
MARKET_TICKERS = {
    "Singapore Healthcare": {
        # --- 病院経営・総合クリニック ---
        "Raffles Medical Group (ラッフルズ医療)": "BSL.SI",  # 【修正】R01.SI から変更
        "IHH Healthcare (マウント・エリザベス等経営)": "Q0F.SI",  # 【修正】I01.SI から変更
        "Thomson Medical Group (トムソン医療)": "A50.SI",
        
        # --- 医療不動産（REIT / 病院・介護施設） ---
        "Parkway Life REIT (パークウェイ・ライフ)": "C2PU.SI",
        "First REIT (ファースト・リート)": "AW9U.SI",
        
        # --- 専門医療・歯科クリニックチェーン ---
        "Q&M Dental Group (歯科大手チェーン)": "QC7.SI",
        "Alliance Healthcare Group (総合診療・クリニック)": "MIJ.SI",  # 【修正】1D8から変更
        "Medinex Limited (医療サポート・クリニック運営)": "OTX.SI",
        
        # --- 医療用消耗品・製薬 ---
        "Medtecs International (医療用消耗品)": "546.SI",
        "Hyphens Pharma (医薬品・皮膚科製品)": "1J5.SI",
        "Riverstone": "AP4.SI"
    }
}
show_market_performances(MARKET_TICKERS)

Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Raffles Medical Group (ラッフルズ医療),BSL.SI,0.94,+0.53%,+1.61%,+1.61%,+0.53%,-2.07%,-2.07%
IHH Healthcare (マウント・エリザベス等経営),Q0F.SI,2.70,-2.88%,-5.26%,-2.17%,-2.17%,-6.25%,-6.25%
Thomson Medical Group (トムソン医療),A50.SI,0.05,+1.89%,+0.00%,+1.89%,-1.82%,-6.90%,-6.90%
Parkway Life REIT (パークウェイ・ライフ),C2PU.SI,4.11,+0.49%,+0.49%,+0.24%,+4.31%,+3.53%,+3.53%
First REIT (ファースト・リート),AW9U.SI,0.23,+0.00%,+0.00%,+0.00%,+0.00%,-4.17%,-4.17%
Q&M Dental Group (歯科大手チェーン),QC7.SI,0.56,+0.00%,+1.83%,+0.91%,-6.72%,-3.48%,-3.48%
Alliance Healthcare Group (総合診療・クリニック),MIJ.SI,0.16,+0.00%,+0.00%,+0.00%,+0.00%,-3.07%,-3.07%
Medinex Limited (医療サポート・クリニック運営),OTX.SI,0.23,+0.00%,+0.00%,+0.00%,+9.52%,+2.22%,+2.22%
Medtecs International (医療用消耗品),546.SI,0.11,+0.88%,+0.00%,-0.87%,-6.56%,-10.94%,-10.94%
Hyphens Pharma (医薬品・皮膚科製品),1J5.SI,0.35,+0.00%,+0.00%,+0.00%,+6.06%,+6.06%,+6.06%


In [4]:
MARKET_TICKERS = {
"Singapore Food & Agribusiness": {
        "Wilmar International (ウィルマー・世界最大級のパーム油)": "F34.SI",
        "Olam Group (オラム・ココア、コーヒー、ナッツ等大手)": "VC2.SI",
        "Riverstone": "AP4.SI",
        "ヤンジジャン・シップビルディング(BS6)": "BS6.SI",
        "Sembcorp Ind (U96)": "U96.SI",
    }
}
show_market_performances(MARKET_TICKERS)

Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Wilmar International (ウィルマー・世界最大級のパーム油),F34.SI,3.85,+1.05%,+2.67%,+3.49%,+7.84%,+1.32%,+1.32%
Olam Group (オラム・ココア、コーヒー、ナッツ等大手),VC2.SI,1.23,+3.36%,+1.65%,+0.82%,-3.91%,+0.82%,+0.82%
Riverstone,AP4.SI,0.86,+1.18%,+1.79%,+0.59%,-1.16%,-8.56%,-8.56%
ヤンジジャン・シップビルディング(BS6),BS6.SI,3.61,+5.87%,+3.74%,+1.40%,+4.64%,-9.30%,-9.30%
Sembcorp Ind (U96),U96.SI,5.68,+1.43%,-0.87%,-5.02%,-7.34%,-8.24%,-8.24%


<div style="background-color: #EBCB8B; color: #2E3440; padding: 12px; border-radius: 6px; font-weight: bold;">
</div>